In [0]:
-- Per-request cost breakdown for databricks-gpt-oss-120b
-- Adjust token rates below as needed (price per 1K tokens)
WITH rates AS (
  SELECT
    0.00075 AS input_rate_per_1k,   -- $/1K input tokens
    0.002   AS output_rate_per_1k   -- $/1K output tokens
)
SELECT
  u.request_time,
  u.requester AS user_name,
  u.input_token_count,
  u.output_token_count,
  u.input_token_count + u.output_token_count AS total_tokens,
  ROUND(u.input_token_count / 1000.0 * r.input_rate_per_1k, 6) AS input_cost,
  ROUND(u.output_token_count / 1000.0 * r.output_rate_per_1k, 6) AS output_cost,
  ROUND(
    u.input_token_count / 1000.0 * r.input_rate_per_1k +
    u.output_token_count / 1000.0 * r.output_rate_per_1k,
    6
  ) AS total_cost
FROM system.serving.endpoint_usage u
JOIN system.serving.served_entities e
  ON u.served_entity_id = e.served_entity_id
CROSS JOIN rates r
WHERE e.endpoint_name = 'databricks-gpt-oss-120b'
ORDER BY u.request_time DESC